In [ ]:
import geopandas as gpd
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm

import numpy as np
import seaborn as sns
from shapely.wkt import loads
import cafo_iowa.db.session as s

from cafo_iowa.estimate.estimate import load_and_process_facilities
from cafo_iowa.utils.visualize import simple_map
from cafo_iowa.utils.utils import extract_geoms_from_list, save_geojson_with_lists

In [ ]:
# load facilities
facilities = load_and_process_facilities(buffer_size=1000)

# load labeled tiles
labeled_tiles = gpd.read_postgis(
    """
    SELECT id, geometry, geometry_buffer
    FROM processed.naip21_qt
    WHERE id IN (
        SELECT jsonb_array_elements_text(naip_qt_ids)
        FROM processed.label_batches
    )
    """,
    con=s.get_engine(),
    geom_col="geometry",
)

In [ ]:
# get all barns
all_barns = gpd.read_postgis(
    """
    SELECT *
    FROM processed.barns
    """,
    con=s.get_engine(),
    geom_col="geometry",
)

# dissolve by cluster id
all_barns_dissolved = all_barns.dissolve(by="barn_cluster_id")

In [ ]:
# NOTE: subset facility data here to only look at certain facilities.
# e.g. swine facilities with no barns
data = facilities.copy()
data = data[(data.barn_count == 0) & (data.animal_cat_combined == "swine")]

crs = data.crs

labeled_tiles = labeled_tiles[labeled_tiles.intersects(data.unary_union)]

In [ ]:
filename = "data_no_barns"
# save as excel
data.to_excel(f"output/data/temp/{filename}.xlsx")

# save as shapefile
save_geojson_with_lists(data, f"output/data/temp/{filename}.geojson")

labeled_tiles.to_file(
    f"output/data/temp/{filename}_labeled_tiles.geojson", driver="GeoJSON"
)

# extract permit ids and geometry
data_permits = (
    data[["permit_ids", "permit_geometries"]]
    .explode(["permit_ids", "permit_geometries"])
    .copy()
)
data_permits = gpd.GeoDataFrame(data_permits, crs=crs, geometry="permit_geometries")
data_permits = data_permits.rename(columns={"permit_ids": "permit_id"})
# save as shapefile
data_permits.to_file(f"output/data/temp/{filename}_permits.geojson", driver="GeoJSON")

all_barns_dissolved.to_file(
    f"output/data/temp/{filename}_all_barns.geojson", driver="GeoJSON"
)

In [ ]:
facility_geoms = data.facility_geom
facility_parcel_geoms = extract_geoms_from_list(
    data, column_name="parcel_geometries", crs=crs
)
facility_permit_geoms = extract_geoms_from_list(
    data, column_name="permit_geometries", crs=crs
)
facility_barn_geoms = extract_geoms_from_list(
    data, column_name="barn_geometries", crs=crs
)

In [ ]:
print(
    f"Facilities: {facility_geoms.shape[0]}\nFacility parcels: {facility_parcel_geoms.shape[0]}\nFacilty permits: {facility_permit_geoms.shape[0]}\nFacility barns: {facility_barn_geoms.shape[0]}\nAll barns: {all_barns.shape[0]}\nAll barn clusters: {all_barns_dissolved.shape[0]}"
)

In [ ]:
data.reported_animal_units.value_counts(sort=True)

In [ ]:
simple_map(
    "EPSG:4326",
    facility_parcel_geoms,
)